In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score


In [6]:
dataset = pd.read_csv('Social_Network_Ads.csv') #dataset collection
dataset = pd.get_dummies(dataset, drop_first = True, dtype = int) #dataset preprocessing by converting the nominal into ordinal 
dataset.drop('User ID' , axis = 1) #we cant put any test input for user id column hence its removed

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1
...,...,...,...,...
395,46,41000,1,0
396,51,23000,1,1
397,50,20000,1,0
398,36,33000,0,1


In [7]:
dataset.columns

Index(['User ID', 'Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [8]:
#separate input and output
input = dataset[['User ID', 'Age', 'EstimatedSalary',  'Gender_Male']]
output = dataset [['Purchased']]


In [9]:
#separate train and test data
x_train, x_test, y_train, y_test = train_test_split(input, output, test_size = 1/3, random_state = 0)

In [10]:
#dataset preprocessing by standardscaler
#standardize the  data by converting the exist value into certain range (0 to 1)
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [11]:
dataset[['Purchased']].value_counts()#dataset is highly imbalanced

Purchased
0            257
1            143
Name: count, dtype: int64

In [12]:
#hypertune by Gridsearch CV
parameter = {'priors':[None], 'var_smoothing': [1e-09]}
grid = GridSearchCV(GaussianNB (), parameter, refit = True, verbose = 3, n_jobs = -1, scoring= 'f1_weighted')
grid.fit(x_train, y_train)

Fitting 5 folds for each of 1 candidates, totalling 5 fits


C:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,estimator,GaussianNB()
,param_grid,"{'priors': [None], 'var_smoothing': [1e-09]}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,priors,None


In [14]:
#choose the best parameter
result = grid.cv_results_
grid_pred = grid.predict(x_test)

cm = confusion_matrix(y_test, grid_pred)
#confusion matrix

clf = classification_report(y_test, grid_pred)
#classification report generate

print(cm)
print(clf)

[[80  5]
 [ 8 41]]
              precision    recall  f1-score   support

           0       0.91      0.94      0.92        85
           1       0.89      0.84      0.86        49

    accuracy                           0.90       134
   macro avg       0.90      0.89      0.89       134
weighted avg       0.90      0.90      0.90       134



In [17]:
#generate f1 score
f1_macro = f1_score(y_test, grid_pred, average = 'weighted')
print("The f1_macro value for best parameter {}.". format(grid.best_params_), f1_macro)

The f1_macro value for best parameter {'priors': None, 'var_smoothing': 1e-09}. 0.9022944298888884


In [20]:
#generate roc_auc_score value
roc = roc_auc_score(y_test, grid.predict_proba(x_test)[:,1])
roc

0.9601440576230492